In [ ]:
%load_ext autoreload
%autoreload 2
# %cd /pscratch/sd/b/brenthu/chem_llm
%cd /nfs/roberts/project/pi_vsb4/byh2/chem_llm

import json
import os
import config
from dotenv import load_dotenv
load_dotenv()
    
# HF_HOME must be set before transformers is imported so it picks up the cache dir.
os.environ["HF_HOME"] = config.HF_HOME
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from agent_core import run_agent

In [2]:
os.makedirs(config.WORK_DIR, exist_ok=True)
os.chdir(config.WORK_DIR)
print(f"Working directory: {os.getcwd()}")

Working directory: /nfs/roberts/project/pi_vsb4/byh2/chem_llm/test9


In [3]:
# LOAD MODEL

if not config.HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is not set. Run `export HF_TOKEN=hf_xxx` before launching main.py."
    )
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, token=config.HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
    token=config.HF_TOKEN,
)

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

In [4]:
# TASK = """
# Write a Python script named generate_structures.py that generates crystal structure CIF files interpolating between the cubic and orthorhombic phases of CsPbBr3.

# Required workflow:
# 1. Use the generate_cif tool to obtain two endpoint structures:
#    - Cubic CsPbBr3 (space group Pm-3m)
#    - Orthorhombic CsPbBr3 (space group Pnma)
# 2. Read both generated CIF files and use them as the structural references for interpolation. Do not hardcode lattice parameters or atomic coordinates from external sources.
# 3. The endpoint structures have different unit cells (the cubic structure has Z=1 while the orthorhombic structure has Z=4). Before interpolating, convert them into compatible representations with the same composition, number of atoms, and atom ordering so that a one-to-one correspondence between atoms can be established.
# 4. Write generate_structures.py using pymatgen. The script should generate 50 intermediate structures by smoothly interpolating both the lattice and atomic positions between the aligned endpoint structures and save them as CIF files.
# 5. After running the script, there should be 52 CIF files total:
#    - 2 endpoint CIFs
#    - 50 interpolated CIFs

# Requirements:
# - Base the interpolation on the endpoint structures obtained from generate_cif.
# - Preserve the composition CsPbBr3 throughout.
# - Organize the output files in a clear directory structure.
# - Comment the code where nontrivial crystallographic operations are performed.

# Before calling done:
# - Execute generate_structures.py.
# - Verify it completes without errors.
# - Read at least one intermediate CIF and confirm it appears reasonable.

# In the done summary, include:
# - The Materials Project IDs used for the endpoint structures.
# - How the two structures were transformed into compatible cells.
# - How atom correspondence was established.
# - The interpolation method used for the lattice and atomic coordinates.
# - Any scientific assumptions or approximations made.
# """

In [5]:
TASK = """
Write two new Python scripts based off of the pre-existing generate_structures.py and setup_jobs.py that generates perturbed TiO2 crystal structures using pymatgen and input files for quantum espresso.

Required workflow:
1. Read the existing generate_structures.py and note down how each constant and function works as well as anything you don't need for TiO2 (for example, interpolating between phase transitions)
2. Use the generate_cif tool to obtain the base rutile TiO4 crystal structure.
3. In the python script, use pymatgen to generate 50 additional structures for 51 total structures.
4. The perturbations should be chemically reasonable random displacements of the Ti and O atoms at room temperature.
5. Then read the existing setup_jobs.py, noting down what it does and any changes you must make to work with the new workflow
6. Then write a new python file. Leave SLURM account and email and other stuff like that blank though

Requirements:
- Base the perturbations on the endpoint structure obtained from generate_cif.
- Organize the output files in a clear directory structure.
- Comment the code where nontrivial crystallographic operations are performed.

Before calling done:
- Execute generate_structures.py and setup_jobs.py.
- Verify it completes without errors.
- Read at least one intermediate CIF and one quantum espresso input file to confirm it appears reasonable.

In the done summary, include:
- The Materials Project IDs used for the endpoint structures.
- An outline of how the new python workflow works.
- Any scientific assumptions made and a quick justification.
- Any details that might require further expert domain knowledge.
"""

In [6]:
final_state = run_agent(TASK, model, tokenizer, verbose=True)

print("\nFINAL STATE:\n", json.dumps(final_state.to_dict(), indent=2))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== STEP 1 ===
RAW MODEL OUTPUT:
 { "tool": "note", "args": { "text": "Starting task: create two new Python scripts for generating perturbed TiO2 structures and Quantum ESPRESSO input files. First, I will analyze the existing generate_structures.py and setup_jobs.py to understand their structure and functionality. Then, I will use generate_cif to obtain the base rutile TiO2 structure from the Materials Project." } }
PARSED TOOL CALL:
 {'tool': 'note', 'args': {'text': 'Starting task: create two new Python scripts for generating perturbed TiO2 structures and Quantum ESPRESSO input files. First, I will analyze the existing generate_structures.py and setup_jobs.py to understand their structure and functionality. Then, I will use generate_cif to obtain the base rutile TiO2 structure from the Materials Project.'}}
TOOL RESULT:
 noted

=== STEP 2 ===
RAW MODEL OUTPUT:
 { "tool": "generate_cif", "args": { "composition": "TiO2", "output_path": "rutile_TiO2.cif", "spacegroup_symbol": "P4_2/mnm

Retrieving SummaryDoc documents:   0%|          | 0/46 [00:00<?, ?it/s]

TOOL RESULT:
 {'success': True, 'selected_material_id': 'mp-2657', 'candidate_material_ids': ['mp-2657'], 'space_group': {'symbol': 'P4_2/mnm', 'number': 136}, 'output_path': '/nfs/roberts/project/pi_vsb4/byh2/chem_llm/test9/rutile_TiO2.cif'}

=== STEP 3 ===
RAW MODEL OUTPUT:
 { "tool": "read_file", "args": { "path": "rutile_TiO2.cif" } }
PARSED TOOL CALL:
 {'tool': 'read_file', 'args': {'path': 'rutile_TiO2.cif'}}
TOOL RESULT:
 {'path': 'rutile_TiO2.cif', 'content': "# generated using pymatgen\ndata_TiO2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   5.46950206\n_cell_length_b   5.46950535\n_cell_length_c   5.46950526\n_cell_angle_alpha   114.49126058\n_cell_angle_beta   107.01845595\n_cell_angle_gamma   107.02372477\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   TiO2\n_chemical_formula_sum   'Ti4 O8'\n_cell_volume   125.22506014\n_cell_formula_units_Z   4\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site

Retrieving SummaryDoc documents:   0%|          | 0/46 [00:00<?, ?it/s]

TOOL RESULT:
 {'success': True, 'selected_material_id': 'mp-2657', 'candidate_material_ids': ['mp-2657'], 'space_group': {'symbol': 'P4_2/mnm', 'number': 136}, 'output_path': '/nfs/roberts/project/pi_vsb4/byh2/chem_llm/test9/rutile_TiO2.cif'}

=== STEP 11 ===
RAW MODEL OUTPUT:
 {
  "tool": "run_python",
  "args": {
    "path": "generate_structures.py"
  }
}
PARSED TOOL CALL:
 {'tool': 'run_python', 'args': {'path': 'generate_structures.py'}}
TOOL RESULT:
 {'stdout': 'Generating 51 TiO2 structures (1 base + 50 perturbed)...\nWrote base structure: structures/structure_000.cif\nWrote perturbed structure: structures/structure_001.cif\nWrote perturbed structure: structures/structure_002.cif\nWrote perturbed structure: structures/structure_003.cif\nWrote perturbed structure: structures/structure_004.cif\nWrote perturbed structure: structures/structure_005.cif\nWrote perturbed structure: structures/structure_006.cif\nWrote perturbed structure: structures/structure_007.cif\nWrote perturbed str